<a href="https://colab.research.google.com/github/DangLeUyen/Reinforcement-Learning/blob/main/U6_A2C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Advantage Actor Critic (A2C) using Robotics Simulations with Panda-Gym

https://huggingface.co/leuyendt/a2c-PandaReachDense-v3/tree/main/


### 1. Create a virtual display

In [1]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

In [2]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 2. Install dependencies

In [3]:
!pip install stable-baselines3[extra]
!pip install gymnasium
!pip install huggingface_sb3
!pip install huggingface_hub
!pip install panda_gym

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.10.2 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873446 sha256=2faf5ce772763415927b906dcff979fe17155b5f3d1ea4f08fbb

In [4]:
# Import the libraries
import os

import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub

from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from huggingface_hub import notebook_login

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Create the environment

In [5]:
env_id = "PandaReachDense-v3"

# Create the env
env = gym.make(env_id)

# Get the state space and action space
s_size = env.observation_space.shape
a_size = env.action_space

In [6]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

_____OBSERVATION SPACE_____ 

The State Space is:  None
Sample observation {'achieved_goal': array([-4.503845 ,  9.874615 ,  4.6584044], dtype=float32), 'desired_goal': array([-6.407009 , -1.0504836, -0.2941508], dtype=float32), 'observation': array([ 5.434121 , -3.9439247,  7.307039 , -7.553211 , -4.286128 ,
       -8.495458 ], dtype=float32)}


In [7]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action


 _____ACTION SPACE_____ 

The Action Space is:  Box(-1.0, 1.0, (3,), float32)
Action Space Sample [-0.43760875 -0.68382883  0.65400416]


## 4. Normalize observation and rewards

In [8]:
env = make_vec_env(env_id, n_envs=4)

# Adding this wrapper to normalize the observation and the reward
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

## 5. Create the A2C Model

In [9]:
model = A2C(policy = "MultiInputPolicy",
            env = env,
            verbose=1)


Using cpu device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
model.learn(1_000_000)


Streaming output truncated to the last 5000 lines.
|    std                | 0.405    |
|    value_loss         | 0.000395 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.81     |
|    ep_rew_mean        | -0.214   |
|    success_rate       | 1        |
| time/                 |          |
|    fps                | 325      |
|    iterations         | 23800    |
|    time_elapsed       | 1462     |
|    total_timesteps    | 476000   |
| train/                |          |
|    entropy_loss       | -1.38    |
|    explained_variance | 0.955    |
|    learning_rate      | 0.0007   |
|    n_updates          | 23799    |
|    policy_loss        | 0.000361 |
|    std                | 0.404    |
|    value_loss         | 0.000144 |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 2.91     |
|    ep_rew_mean        

In [11]:
# Save the model and  VecNormalize statistics when saving the agent
model.save("a2c-PandaReachDense-v3")
env.save("vec_normalize.pkl")


## 6. Evaluate the agent

In [12]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Load the saved statistics
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

# We need to override the render_mode
eval_env.render_mode = "rgb_array"

#  do not update them at test time
eval_env.training = False
# reward normalization is not needed at test time
eval_env.norm_reward = False

# Load the agent
model = A2C.load("a2c-PandaReachDense-v3")

mean_reward, std_reward = evaluate_policy(model, eval_env)

print(f"Mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

Mean reward = -0.21 +/- 0.15


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


## 7. Publish to the Hub

In [ ]:
notebook_login()
!git config --global credential.helper store


In [ ]:
from huggingface_sb3 import package_to_hub
from stable_baselines3.common.monitor import Monitor
# from stable_baselines3.common.vec_env import VecVideoRecorder # Removed explicit VecVideoRecorder

# Define a function to create a single base environment
def make_env_for_eval():
    env = gym.make(env_id, render_mode="rgb_array")
    # Monitor for logging stats during evaluation
    return Monitor(env)

# Create DummyVecEnv with the base environment
temp_eval_env_for_norm = DummyVecEnv([make_env_for_eval])

# Load VecNormalize statistics
eval_env_normalized = VecNormalize.load("vec_normalize.pkl", temp_eval_env_for_norm)
eval_env_normalized.training = False
eval_env_normalized.norm_reward = False

# Pass the normalized environment directly to package_to_hub.
# Let package_to_hub handle its own video recording logic.
package_to_hub(
    model=model,
    model_name=f"a2c-{env_id}",
    model_architecture="A2C",
    env_id=env_id,
    eval_env=eval_env_normalized, # Pass the normalized env directly, without VecVideoRecorder
    repo_id=f"leuyendt/a2c-{env_id}", # Change the username
    commit_message="Initial commit",
)